Vendas de carros, em dólares. 



'make', 'model', 'year', 'mileage(miles)', 'engine_hp', 'transmission',
'fuel_type', 'drivetrain', 'body_type', 'exterior_color',
'interior_color', 'owner_count', 'accident_history', 'seller_type',
'condition', 'trim', 'vehicle_age', 'mileage_per_year', 'price(dolar)'

# Imports

In [1]:
import pandas as pd
import sqlite3

# Start

Dados retirados do kaggle
https://www.kaggle.com/datasets/metawave/vehicle-price-prediction

In [2]:
df = pd.read_csv('../data/price_cars.csv')
df = df.head(250000).drop(columns=['brand_popularity'])
df_new = df.tail(2000)


In [3]:
conn = sqlite3.connect('../sql/price_cars.db')
df.to_sql('cars', conn, if_exists='replace', index=False)
df_new.to_sql('cars_new', conn, if_exists='replace', index=False)

query = 'SELECT * ' \
'FROM cars ' \
'LIMIT 5'

pd.read_sql(query, conn)



,make,model,year,mileage,engine_hp,transmission,fuel_type,drivetrain,body_type,exterior_color,interior_color,owner_count,accident_history,seller_type,condition,trim,vehicle_age,mileage_per_year,price
0,Volkswagen,Jetta,2016,183903,173,Manual,Electric,RWD,Sedan,Blue,Brown,5,None,Dealer,Excellent,EX,9,20433.666667,7208.52
1,Lexus,RX,2010,236643,352,Manual,Gasoline,FWD,Sedan,Silver,Beige,5,Minor,Dealer,Good,LX,15,15776.200000,6911.81
2,Subaru,Crosstrek,2016,103199,188,Automatic,Diesel,AWD,Sedan,Silver,Beige,5,None,Dealer,Excellent,Touring,9,11466.555556,11915.63
3,Cadillac,Lyriq,2016,118889,338,Manual,Gasoline,AWD,SUV,Black,Gray,3,None,Private,Good,Base,9,13209.888889,25984.79
4,Toyota,Highlander,2018,204170,196,Manual,Diesel,FWD,Sedan,Red,Brown,5,Minor,Dealer,Excellent,Sport,7,29167.142857,8151.30


In [4]:
cursor= conn.cursor()

# Verificando os dados

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   make              250000 non-null  object 
 1   model             250000 non-null  object 
 2   year              250000 non-null  int64  
 3   mileage           250000 non-null  int64  
 4   engine_hp         250000 non-null  int64  
 5   transmission      250000 non-null  object 
 6   fuel_type         250000 non-null  object 
 7   drivetrain        250000 non-null  object 
 8   body_type         250000 non-null  object 
 9   exterior_color    250000 non-null  object 
 10  interior_color    250000 non-null  object 
 11  owner_count       250000 non-null  int64  
 12  accident_history  62513 non-null   object 
 13  seller_type       250000 non-null  object 
 14  condition         250000 non-null  object 
 15  trim              250000 non-null  object 
 16  vehicle_age       25

# Sobre os dados

Antes de fazer alterações foi feita uma primeira exploração dos dados. Analisando os resultados foi possível identificar informações que não condizem com a realidade, todas as proporções resultam em 50/50 ou 33.3/33.3/33.3 etc, isso cria a hipótese de que os dados foram gerados artificialmente. Olhando para a descrição da página onde se encontra esse dataset, foi confirmado que os dados são sintéticos.

Também foi possível identificar algumas outras inconsistências com a realidade, por exemplo a existência de carros manuais da marca Tesla, ou veículos dessa mesma marca em 2003 sendo que a marca lançou seu primeiro carro elétrico em 2008.

Para que os dados reflitam com a realidade, seria necessária uma análise mais profunda em cada modelo para verificar tipos de transmissão, combustível, tração, etc

As mudanças necessárias serão aplicadas no SELECT para criar o CSV dos dados.

OBS: As regras de remoção foram feitas especialmente para este dataset, levando em conta que não haverá inputs de marcas, modelos ou tipo de combustível novos.

In [6]:
cursor.execute(
    '''
    UPDATE cars
    SET accident_history='No accident'
    WHERE accident_history IS NULL
    ''')
conn.commit()

In [7]:
## tabela para ser usada como dados novos
cursor.execute(
    '''
    UPDATE cars_new
    SET accident_history='No accident'
    WHERE accident_history IS NULL
    ''')
conn.commit()

## Nova tabela com país de origem de cada marca

In [8]:
cursor.execute('''
               CREATE TABLE IF NOT EXISTS brand_origin 
               (
                    make TEXT PRIMARY KEY,
                    country TEXT
               )
               ''')

brands = [
    ('Toyota','Japan'), ('Nissan', 'Japan'),
    ('Mazda', 'Japan'), ('Honda', 'Japan'),
    ('Subaru', 'Japan'), ('Lexus', 'Japan'),
    ('Acura', 'Japan'), ('Volkswagen', 'Germany'),
    ('Audi', 'Germany'), ('BMW', 'Germany'),
    ('Mercedes-Benz', 'Germany'), ('Porsche', 'Germany'),
    ('Ford', 'USA'), ('Chevrolet', 'USA'),
    ('Tesla', 'USA'), ('Jeep', 'USA'),
    ('Dodge', 'USA'), ('Chrysler', 'USA'),
    ('GMC', 'USA'), ('Ram', 'USA'),
    ('Cadillac', 'USA'), ('Kia', 'South Korea'),
    ('Hyundai', 'South Korea'), ('Volvo', 'Sweden'),
    ('Land Rover', 'UK'),
]
cursor.executemany('INSERT OR IGNORE INTO brand_origin VALUES(?,?)', brands)
conn.commit()


# Queries

In [9]:
## preview
query = ('''
    SELECT * 
    FROM cars
    LIMIT 5; 
''')

pd.read_sql(query, conn)

,make,model,year,mileage,engine_hp,transmission,fuel_type,drivetrain,body_type,exterior_color,interior_color,owner_count,accident_history,seller_type,condition,trim,vehicle_age,mileage_per_year,price
0,Volkswagen,Jetta,2016,183903,173,Manual,Electric,RWD,Sedan,Blue,Brown,5,No accident,Dealer,Excellent,EX,9,20433.666667,7208.52
1,Lexus,RX,2010,236643,352,Manual,Gasoline,FWD,Sedan,Silver,Beige,5,Minor,Dealer,Good,LX,15,15776.200000,6911.81
2,Subaru,Crosstrek,2016,103199,188,Automatic,Diesel,AWD,Sedan,Silver,Beige,5,No accident,Dealer,Excellent,Touring,9,11466.555556,11915.63
3,Cadillac,Lyriq,2016,118889,338,Manual,Gasoline,AWD,SUV,Black,Gray,3,No accident,Private,Good,Base,9,13209.888889,25984.79
4,Toyota,Highlander,2018,204170,196,Manual,Diesel,FWD,Sedan,Red,Brown,5,Minor,Dealer,Excellent,Sport,7,29167.142857,8151.30


In [10]:
## Quais são as marcas?
query = ('''
    SELECT DISTINCT make 
     FROM cars;
     ''' )

print(list(pd.read_sql(query, conn)['make']))

['Volkswagen', 'Lexus', 'Subaru', 'Cadillac', 'Toyota', 'Land Rover', 'Mazda', 'Ram', 'Chrysler', 'GMC', 'Volvo', 'Audi', 'Chevrolet', 'Tesla', 'Hyundai', 'Ford', 'Porsche', 'Acura', 'Nissan', 'Kia', 'Jeep', 'BMW', 'Dodge', 'Mercedes-Benz', 'Honda']


In [11]:
## Quantas marcas?
query = ('''
    SELECT COUNT(DISTINCT make) \
     FROM cars; \
''')

pd.read_sql(query, conn)

,COUNT(DISTINCT make)
0,25


In [12]:
## Quais são as 10 marcas com maior volume de carros à venda?
query = ('''
SELECT make, COUNT(make) AS qtd, ROUND(COUNT(make)*100.0/(SELECT COUNT(*) FROM cars),3) AS percentage 
FROM cars  
GROUP BY make  
ORDER BY qtd DESC  
LIMIT 10 
''')

pd.read_sql(query, conn)

,make,qtd,percentage
0,Mazda,10197,4.079
1,Honda,10195,4.078
2,Kia,10166,4.066
3,Land Rover,10165,4.066
4,Audi,10118,4.047
5,Ram,10060,4.024
6,Porsche,10046,4.018
7,Subaru,10036,4.014
8,GMC,10028,4.011
9,BMW,10018,4.007


In [13]:
## Qual o preço médio de venda por tipo de combustível?
query = ('''
SELECT fuel_type, AVG(price) AS mean_price_per_fuel_type  
FROM cars  
GROUP BY fuel_type 
''')

pd.read_sql(query, conn)

,fuel_type,mean_price_per_fuel_type
0,Diesel,19799.531901
1,Electric,21257.983922
2,Gasoline,19840.552877


In [14]:
## Qual a distribuição de carros por tipo de transmissão (manual vs automático)?
query =('''
SELECT transmission, COUNT(transmission) AS qtd, ROUND(COUNT(transmission)*100.0/(SELECT COUNT(*) FROM cars),3) AS percentage 
FROM cars 
GROUP BY transmission
''')

pd.read_sql(query, conn)

,transmission,qtd,percentage
0,Automatic,125175,50.07
1,Manual,124825,49.93


In [15]:
## Antes de criar a tabela de brands
# ['Volkswagen', 'Lexus', 'Subaru', 'Cadillac', 'Toyota', 'Land Rover', 'Mazda', 'Ram', 'Chrysler', 'GMC', 'Volvo',
#  'Audi', 'Chevrolet', 'Tesla', 'Hyundai', 'Ford', 'Porsche', 'Acura', 'Nissan', 'Kia', 'Jeep', 'BMW', 'Dodge', 'Mercedes-Benz', 'Honda']
## Quantidade por país
query = ('''
    WITH with_country AS 
        (
        SELECT make, mileage, engine_hp, drivetrain, seller_type, condition, vehicle_age, price,
         CASE 
            WHEN make in ("Volkswagen", "Audi", "BMW", "Mercedes-Benz", "Porsche") THEN "Germany"
            WHEN make in ("Ford", "Chevrolet", "Tesla", "Jeep", "Dodge", "Chrysler", "GMC", "Ram", "Cadillac") THEN "USA" 
            WHEN make in ("Toyota", "Mazda", "Nissan", "Subaru", "Lexus", "Acura", "Honda") THEN "Japan" 
            WHEN make in ("Hyundai", "Kia") THEN "South Korea" 
            WHEN make in ("Volvo") THEN "Sweden" 
            WHEN make in ("Land Rover") THEN "UK" 
         END AS country
        FROM cars 
        )
    SELECT country, ROUND(AVG(price),2) AS mean_price
    FROM with_country
    GROUP BY country
    ORDER BY mean_price DESC 
''')

pd.read_sql(query, conn)

,country,mean_price
0,UK,39506.71
1,Germany,27557.12
2,Sweden,27156.97
3,USA,19978.58
4,Japan,14880.19
5,South Korea,10037.17


In [16]:
## Top 10 países que mais vende
query = ('''
    SELECT b.country, COUNT(b.country)
     FROM cars AS c 
     JOIN brand_origin AS b ON c.make = b.make
     GROUP BY b.country
     ORDER BY COUNT(b.country) DESC
''')

pd.read_sql(query, conn)

,country,COUNT(b.country)
0,USA,89597
1,Japan,70363
2,Germany,49947
3,South Korea,20074
4,UK,10165
5,Sweden,9854


In [17]:
## quantidade de modelos por marca
query = ('''
    SELECT make, COUNT(DISTINCT model) AS qtd_model
     FROM cars 
     GROUP BY make
''')

pd.read_sql(query, conn)

,make,qtd_model
0,Acura,4
1,Audi,5
2,BMW,5
3,Cadillac,4
4,Chevrolet,5
5,Chrysler,2
6,Dodge,3
7,Ford,5
8,GMC,4
9,Honda,5


In [18]:
## quantidade de modelos por país
query = ('''
    SELECT b.country , COUNT(DISTINCT c.model) AS qtd_models 
     FROM cars AS c JOIN brand_origin AS b ON c.make = b.make 
     GROUP BY b.country 
     ORDER BY COUNT(DISTINCT c.model) DESC
''')

pd.read_sql(query, conn)

,country,qtd_models
0,USA,34
1,Japan,32
2,Germany,23
3,South Korea,9
4,Sweden,4
5,UK,3


In [19]:
## Quais marcas têm a melhor relação entre preço e potência (selling_price / max_power)?
query = ('''
SELECT make, price/engine_hp AS price_per_hp  
FROM cars  
GROUP BY make  
ORDER BY price_per_hp ASC  
LIMIT 10
''')

pd.read_sql(query, conn)

,make,price_per_hp
0,Lexus,19.635824
1,BMW,22.382417
2,Honda,25.134054
3,Chevrolet,36.590881
4,Acura,37.214063
5,Toyota,41.588265
6,Volkswagen,41.667746
7,GMC,48.604032
8,Ram,58.023604
9,Subaru,63.381011


In [20]:
## Como a idade do carro impacta o preço médio de venda?
query = ('''
    SELECT AVG(vehicle_age) AS avg_vehicle_age
    ,AVG(year) AS avg_year 
    ,AVG(price) AS avg_price
    ,MIN(price) AS min_price
    ,MAX(price) AS max_price
    ,CASE  
        WHEN vehicle_age>=0 AND vehicle_age<=5 THEN "0-5"  
        WHEN vehicle_age>5 AND vehicle_age<=10 THEN "6-10"  
        WHEN vehicle_age>10 AND vehicle_age<=15 THEN "11-15"  
        WHEN vehicle_age>15 AND vehicle_age<=20 THEN "16-20" 
        WHEN vehicle_age>20 THEN "20+" 
    END AS age_range  
    FROM cars  
    GROUP BY age_range 
    ORDER BY avg_vehicle_age 
''')

pd.read_sql(query, conn)

## comportamento estranho no min_price

,avg_vehicle_age,avg_year,avg_price,min_price,max_price,age_range
0,3.194539,2021.933202,31660.591236,7704.9,90693.67,0-5
1,7.941331,2017.058669,18361.272580,1500.0,62086.44,6-10
2,12.371479,2012.628521,9455.601929,1500.0,42926.16,11-15
3,16.932020,2008.067980,4289.693945,1500.0,27682.70,16-20
4,21.582192,2003.417808,2174.776575,1500.0,13317.12,20+


In [21]:
## Vendedores individuais praticam preços menores que concessionárias? (comparar seller_type
query = ('''
    SELECT seller_type
    , AVG(price) AS mean_price
    , COUNT(*) AS count_qtd_sells
    , ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM cars),3) AS percentage_sells 
     FROM cars 
     GROUP BY seller_type
''')

pd.read_sql(query, conn)

,seller_type,mean_price,count_qtd_sells,percentage_sells
0,Dealer,20640.554524,174771,69.908
1,Private,19629.898878,75229,30.092


In [22]:
## Quais modelos desvalorizam menos com o tempo?
## Levando em consideração que não há nenhum modelo que é considerado raro e o valor seja alto
## mesmo que o ano de fabricação seja antigo
query = ('''
    SELECT make, model, MIN(year), MAX(year), MIN(price), MAX(price), MAX(price)-MIN(price) AS diff 
     FROM cars 
     GROUP BY model 
     ORDER BY diff DESC 
     LIMIT 15
''')

pd.read_sql(query, conn)

,make,model,MIN(year),MAX(year),MIN(price),MAX(price),diff
0,Porsche,911,2003,2025,4351.85,90693.67,86341.82
1,Porsche,Panamera,2002,2025,4157.77,88743.78,84586.01
2,Porsche,Cayenne,2004,2025,6802.90,89961.19,83158.29
3,Land Rover,Range Rover,2000,2025,1500.00,84308.56,82808.56
4,Porsche,Macan,2004,2025,6221.68,88840.82,82619.14
5,Land Rover,Defender,2000,2025,1500.00,82630.97,81130.97
6,Land Rover,Discovery,2001,2025,4538.34,82455.17,77916.83
7,Tesla,Model S,2003,2025,1500.00,70256.99,68756.99
8,Tesla,Model Y,2004,2025,3224.40,71400.60,68176.20
9,Tesla,Model X,2005,2025,1500.00,68558.79,67058.79


In [23]:
## Ranking dos 3 modelos mais vendidos dentro de cada marca
query = ('''
    WITH table_ranking AS (
        SELECT make,
        model,
        COUNT(model) AS qtd,
        DENSE_RANK() OVER (PARTITION BY make ORDER BY COUNT(model) DESC) AS ranking 
        FROM cars 
        GROUP BY make, model
    )
    SELECT *
    FROM table_ranking
    WHERE ranking < 4
''' )

pd.read_sql(query, conn)

,make,model,qtd,ranking
0,Acura,RDX,2561,1
1,Acura,TLX,2515,2
2,Acura,Integra,2486,3
3,Audi,Q7,2070,1
4,Audi,Q5,2064,2
...,...,...,...,...
70,Volkswagen,Atlas,1998,2
71,Volkswagen,Jetta,1983,3
72,Volvo,XC90,2524,1
73,Volvo,V60,2475,2


In [24]:
## Relação dos tipos de combustível
query = ('''
     SELECT make, 
     fuel_type,
     COUNT(*) AS qtd,
     ROUND(COUNT(fuel_type) * 100.0/ SUM(COUNT(fuel_type)) OVER(PARTITION BY make),3) AS percentage 
     FROM cars 
     GROUP BY make, fuel_type
''')

pd.read_sql(query, conn)

,make,fuel_type,qtd,percentage
0,Acura,Diesel,3316,33.137
1,Acura,Electric,3340,33.377
2,Acura,Gasoline,3351,33.487
3,Audi,Diesel,3341,33.020
4,Audi,Electric,3370,33.307
...,...,...,...,...
68,Volkswagen,Electric,3363,33.922
69,Volkswagen,Gasoline,3206,32.338
70,Volvo,Diesel,3318,33.672
71,Volvo,Electric,3261,33.093


In [25]:
## Para cada marca, qual o percentual de carros automáticos vs manuais?
query = ('''
     SELECT make, 
    transmission,
    ROUND(COUNT(transmission) * 100.0/ SUM(COUNT(transmission)) OVER(PARTITION BY make),3) AS percentage 
     FROM cars 
     GROUP BY make, transmission
''')

pd.read_sql(query, conn)

,make,transmission,percentage
0,Acura,Automatic,50.884
1,Acura,Manual,49.116
2,Audi,Automatic,50.109
3,Audi,Manual,49.891
4,BMW,Automatic,50.349
5,BMW,Manual,49.651
6,Cadillac,Automatic,50.719
7,Cadillac,Manual,49.281
8,Chevrolet,Automatic,49.100
9,Chevrolet,Manual,50.900


In [26]:
## Qual a quilometragem média dos carros vendidos por concessionárias vs vendedores individuais, segmentado por faixa de preço?
query = ('''
    SELECT seller_type, 
    avg(mileage),
    CASE
       WHEN price<=10000 THEN "0-10k" 
       WHEN price<=20000 AND price>10000 THEN "10k-20k" 
       WHEN price<=30000 AND price>20000 THEN "20k-30k" 
       WHEN price<=40000 AND price>30000 THEN "30k-40k" 
       WHEN price<=50000 AND price>40000 THEN "40k-50k" 
       WHEN price<=60000 AND price>50000 THEN "50k-60k" 
       WHEN price<=70000 AND price>60000 THEN "60k-70k" 
       WHEN price<=80000 AND price>70000 THEN "70k-80k" 
       WHEN price>80000 THEN "80k+" 
    END AS price_range
    FROM cars
    GROUP BY seller_type, price_range
    ORDER BY avg(mileage) DESC 
''')

pd.read_sql(query, conn)

,seller_type,avg(mileage),price_range
0,Dealer,188213.159577,0-10k
1,Private,185223.749615,0-10k
2,Dealer,114639.149589,10k-20k
3,Private,111293.184481,10k-20k
4,Dealer,83450.874218,20k-30k
5,Private,81938.966464,20k-30k
6,Dealer,69811.215163,30k-40k
7,Private,67347.762566,30k-40k
8,Dealer,53296.679919,40k-50k
9,Private,49401.976053,40k-50k


In [27]:
## analisando mileage
query = (
    '''
    SELECT make, model, year, mileage, price
    FROM cars
    ORDER BY mileage ASC, year ASC
    '''
)

pd.read_sql(query, conn)

,make,model,year,mileage,price
0,Audi,R8,2012,500,22475.09
1,Volvo,XC90,2013,500,24185.29
2,Ford,Escape,2013,500,9809.23
3,Acura,RDX,2013,500,16103.95
4,Hyundai,Santa Fe,2014,500,10957.84
...,...,...,...,...,...
249995,Chevrolet,Camaro,2016,300000,2625.98
249996,Ram,2500,2016,300000,5008.68
249997,Audi,Q7,2016,300000,13846.51
249998,Honda,Civic,2016,300000,1500.00


# SQL to csv

In [28]:
query = ('''
               SELECT c.*, b.country
               FROM cars AS c JOIN brand_origin AS b USING (make)
               WHERE NOT
               (
                    (make='Tesla' AND (year<2008 OR fuel_type='Diesel' OR fuel_type='Gasoline' OR transmission='Manual'))
                    OR (model='Defender' AND fuel_type='Electric')
                    OR (make in ('Lexus','Acura','Subaru') AND fuel_type='Diesel')
                    OR
                    ((fuel_type='Electric') AND 
                    (make!='Tesla') AND 
                         (
                              (make='BMW' AND year<2013) OR 
                              (make='Nissan' AND year<2010) OR 
                              (make='Volkswagen' AND year<2013) OR 
                              (make='Hyundai' AND year<2015) OR
                              (make='Kia' AND year<2015) OR
                              (make='Ford' AND year<2016) OR
                              (make='Volvo' AND year<2019) OR 
                              (make='Porsche' AND year<2019)
                         )    
                    )
                    OR (make='Porsche' AND fuel_type='Electric' AND model!='Macan') 
                    OR 
                    ((transmission='Manual') AND 
                         (
                              (make='Chrysler' AND year>2016) OR
                              (make='Lexus' AND year>2013) OR
                              (make='Cadillac' AND year>2019) OR
                              (make='Land Rover' AND year>2019) OR
                              (make='Volvo' AND year>2021)
                         )
                    )
                    OR (mileage_per_year>35000)
               )
          ''')

pc_cleaned = pd.read_sql(query, conn)
pc_cleaned.to_csv('../data/pc_cleaned.csv', index=False)

In [29]:
## query para o dataset que vai ser usado como dados novos
query = ('''
               SELECT c.*, b.country
               FROM cars_new AS c JOIN brand_origin AS b USING (make)
               WHERE NOT
               (
                    (make='Tesla' AND (year<2008 OR fuel_type='Diesel' OR fuel_type='Gasoline' OR transmission='Manual'))
                    OR (model='Defender' AND fuel_type='Electric')
                    OR (make in ('Lexus','Acura','Subaru') AND fuel_type='Diesel')
                    OR
                    ((fuel_type='Electric') AND 
                    (make!='Tesla') AND 
                         (
                              (make='BMW' AND year<2013) OR 
                              (make='Nissan' AND year<2010) OR 
                              (make='Volkswagen' AND year<2013) OR 
                              (make='Hyundai' AND year<2015) OR
                              (make='Kia' AND year<2015) OR
                              (make='Ford' AND year<2016) OR
                              (make='Volvo' AND year<2019) OR 
                              (make='Porsche' AND year<2019)
                         )    
                    )
                    OR (make='Porsche' AND fuel_type='Electric' AND model!='Macan') 
                    OR 
                    ((transmission='Manual') AND 
                         (
                              (make='Chrysler' AND year>2016) OR
                              (make='Lexus' AND year>2013) OR
                              (make='Cadillac' AND year>2019) OR
                              (make='Land Rover' AND year>2019) OR
                              (make='Volvo' AND year>2021)
                         )
                    )
                    OR (mileage_per_year>35000)
               )
          ''')

pc_cleaned = pd.read_sql(query, conn)
pc_cleaned.to_csv('../data/new_data.csv', index=False)
